# Algorithm 1 (APIC) — Snell–Descartes experiment

This notebook reproduces the Snell–Descartes results from Figs. 4(A–C) of the main text using the full iterative forward–backward optimization loop (Algorithm 1).

Each stigmergic cycle consists of:
1. **Forward pass** (`simulate_forward_batch`): generate stochastic trajectories from source to target under the current pheromone field (Eq. 4 of the main text, with trail-following Eq. 2).
2. **Adjoint pass** (`integrate_costate`): integrate the costate equations (Eq. 6) backward along each forward trajectory.
3. **Controlled backward pass** (`simulate_controlled_backward_pass`): generate controlled return trajectories from target to source using the optimal control $\omega_{\mathrm{ctrl}} = -\Gamma/\gamma$ (Eq. 7).
4. **Pheromone update** (`downsample_recent_weighted_trajectories`): deposit pheromone along the controlled backward trajectories with recency weighting.

All five steps are implemented in `src/apic.py` and orchestrated by `run_apic_loop`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import jax
import jax.numpy as jnp
from jax import random
import matplotlib.pyplot as plt

from src.apic import (
    run_apic_loop,
    smooth_piecewise_nu,
    make_init_fn,
    compute_sin_ratio,
)

## Setup

Source at $(0,0)$, target at $(1,1)$, refractive index $\nu=1$ for $y<0.5$ and $\nu=10$ for $y>0.5$ (piecewise sigmoid with steepness 100).

In [ ]:
# Simulation parameters
num_cycles = 5
batch_size = 32
num_steps = 1520
dt = 0.001
sigma_noise = 1.0  # angular noise (D_theta) — controls beta/(l_0 D_theta)
pher_sigma = 0.05  # pheromone kernel width (sigma_trail/l_0)

point_a = jnp.array([0.0, 0.0])
point_b = jnp.array([1.0, 1.0])

init_fn = make_init_fn(point_a, point_b, batch_size)

forward_params  = {"dt": dt, "num_steps": num_steps, "sigma_noise": sigma_noise}
backward_params = {"dt": dt, "num_steps": num_steps, "sigma_noise": sigma_noise}

## Run Algorithm 1 (the full forward-backward loop)

In [ ]:
all_forward, all_backward, pher_pts, pher_wts = run_apic_loop(
    num_cycles=num_cycles,
    key=random.PRNGKey(0),
    init_fn=init_fn,
    point_a=point_a,
    point_b=point_b,
    pher_sigma=pher_sigma,
    forward_params=forward_params,
    backward_params=backward_params,
)

## Verify Snell's law

The refractive index ratio is $\nu_2/\nu_1 = 10$. The Snell ratio $\sin\theta_1/\sin\theta_2$ should approach this value as the stigmergic loop converges.

In [ ]:
results = compute_sin_ratio(all_forward[-1])
valid = ~jnp.isnan(results)
print(f"Mean Snell ratio (cycle {num_cycles}):  {float(jnp.nanmean(results)):.4f}")
print(f"Expected (nu_2 / nu_1):            {float(smooth_piecewise_nu(0,1)/smooth_piecewise_nu(0,0)):.4f}")

## Visualize trajectories across cycles

Reproduces the qualitative behavior of Fig. 4(B): forward trajectories sharpen toward the Snell-optimal refracted path with each stigmergic cycle.

In [ ]:
fig, axes = plt.subplots(1, num_cycles, figsize=(3*num_cycles, 3), sharex=True, sharey=True)
for k, ax in enumerate(axes):
    trajs = all_forward[k]
    for b in range(trajs.shape[1]):
        ax.plot(trajs[:, b, 0], trajs[:, b, 1], color='red', alpha=0.3, lw=0.5)
    ax.axhline(0.5, color='k', ls='--', lw=0.5)
    ax.plot(*point_a, 'o', color='blue')
    ax.plot(*point_b, 'o', color='green')
    ax.set_title(f"Cycle {k}")
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1)
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()